In [ ]:
import math
from dataclasses import replace

import torch
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import hsv_to_rgb

from pathtracer import Renderer, lattice_preview

from spectral import lattice_geometry, row_twiddle, fft, ifft, bin_freq, alias_radius_sq

In [ ]:
def device_mapper() -> torch.device:
    if torch.cuda.is_available(): return torch.device("cuda")
    if torch.backends.mps.is_available(): return torch.device("mps")
    return torch.device("cpu")

def host(x: torch.Tensor) -> np.ndarray: return x.cpu().numpy()

DEVICE = device_mapper()

In [ ]:
scene = Renderer("scenes/cornell.json")
rendercam = replace(scene.default_camera)
halves = torch.from_numpy(np.stack([scene.render(camera=rendercam, aovs=("luminance",), width=64, height=32, samples=scene.default_samples // 2, seed=seed, show_sky=True)["luminance"][..., 0] for seed in (1, 2)])).to(DEVICE)

In [ ]:
plt.imshow(lattice_preview(host(halves.mean(0)[..., None]), rendercam, scale=8))
plt.xticks([]), plt.yticks([])
plt.tight_layout()

In [ ]:
row_pitch, row_shift = lattice_geometry(rendercam)
twiddle = row_twiddle(halves.shape[-2:], row_shift, DEVICE)
fy, fx = bin_freq(halves.shape[-2:], row_pitch, DEVICE)
radius = alias_radius_sq(fy, fx, row_pitch, row_shift)[0].sqrt()
ac = radius > 0
lowest = 1 / min(halves.shape[-1], halves.shape[-2] * row_pitch)
ring = (radius / lowest).long()
count = torch.zeros(int(ring.max()) + 1, device=DEVICE).index_add_(0, ring.flatten(), ac.flatten().float())
ring_mean = lambda v: torch.zeros_like(count).index_add_(0, ring.flatten(), torch.where(ac, v, 0).flatten()) / count

In [ ]:
# luminance > log > noise-aware whitening > oriented filter bank (Gabor or learned) → sparsifying step
log_x = torch.log(halves)

In [ ]:
first, other = fft(log_x, twiddle)
signal = ring_mean((first * other.conj()).real).clamp(min=0)
noise = ring_mean((first - other).abs() ** 2 / 4)
transfer = torch.where(ac, (signal.sqrt() / (signal + noise))[ring], 0)
whitened = (first + other) / 2 * transfer

In [ ]:
gain = signal.sqrt() / (signal + noise)
edges = torch.arange(count.numel() + 1, device=DEVICE) * lowest
centres = (edges[1:] + edges[:-1]) / 2
inside = edges[1:] <= 0.5 * min(1 / row_pitch, math.hypot(1, row_shift / row_pitch))
wavelength = (1 / centres[inside][gain[inside].argmax()]).item()
rows, cols = 2 * math.ceil(wavelength / (2 * row_pitch)), math.ceil(wavelength)
field = torch.roll(ifft(transfer, twiddle).real, (rows, cols), dims=(0, 1))[:2 * rows + 1, :2 * cols + 1]

previews = [lattice_preview(host(x[..., None]), rendercam) for x in (ifft(whitened, twiddle).real, field)]
previews[1] = previews[1][:, :round(previews[1].shape[1] * (1 - row_shift / field.shape[1]))]
fig, ax = plt.subplots(1, 3, figsize=[24, 5], width_ratios=[*(p.shape[1] / p.shape[0] for p in previews), 1])
for a, p, name in zip(ax, previews, ("Whitened", "Receptive field")):
    limit = np.abs(p).max()
    a.imshow(p, cmap="gray", vmin=-limit, vmax=limit)
    a.set_title(name)
    a.set_xticks([]), a.set_yticks([])

ax[2].loglog(host(centres[inside]), host(signal[inside]), lw=1, label="Signal R")
ax[2].loglog(host(centres[inside]), host(noise[inside]), lw=1, label="Noise N")
ax[2].set_title("Signal and noise")
ax[2].set_xlabel("Radius (cycles/pitch)"), ax[2].set_ylabel("Power")
ax[2].legend()
ax[2].set_box_aspect(1), ax[2].grid(alpha=0.25)


In [ ]:
radius_sq, true_fy, true_fx = alias_radius_sq(fy, fx, row_pitch, row_shift)
radius, theta = radius_sq.sqrt(), torch.atan2(true_fy, true_fx)
nyquist = 0.5 * min(1 / row_pitch, math.hypot(1, row_shift / row_pitch))

In [ ]:
sigma_radius = math.log(2) / math.sqrt(8 * math.log(2))
sigma_angle = 1 / math.sqrt(16 * math.log(2))
orientations = math.ceil(math.pi * math.sqrt(2))
scales = math.floor(math.log2(nyquist / lowest))
centre = nyquist * 2 ** -(torch.arange(scales, device=DEVICE) + 0.5)
orientation = torch.arange(orientations, device=DEVICE) * math.pi / orientations
offset = torch.remainder(theta - orientation[:, None, None] + math.pi, 2 * math.pi) - math.pi
radial = torch.exp(-torch.log(radius / centre[:, None, None]) ** 2 / (2 * sigma_radius ** 2))
bank = (radial[:, None] * torch.exp(-offset ** 2 / (2 * sigma_angle ** 2))).flatten(0, 1)

In [ ]:
response = ifft(whitened * bank, twiddle)

In [ ]:
fig, ax = plt.subplots(scales, orientations, figsize=[4 * orientations, 2 * scales], squeeze=False)
for a, band in zip(ax.flat, response.abs()):
    a.imshow(lattice_preview(host(band[..., None]), rendercam), cmap="gray")
    a.set_xticks([]), a.set_yticks([])